In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import glob
import os
from sklearn.model_selection import GroupKFold

base_path = '/kaggle/input/competitions/rogii-wellbore-geology-prediction'

# -------------------------------------------------------------
# ★ メダルに向けた新兵器その2：波のトレンド（ローリング特徴量）を作る関数
# -------------------------------------------------------------
def create_features(df):
    # 掘り進めた順（MD：掘削長）に並べ替えることが超重要
    df = df.sort_values('MD').reset_index(drop=True)
    
    # 1. 前の地点からの変化量（Diff）: 急激にガンマ線が上がったか？
    df['GR_diff'] = df['GR'].diff().fillna(0)
    df['Z_diff'] = df['Z'].diff().fillna(0)
    
    # 2. 直近10ステップの移動平均とばらつき（細かな波の形）
    df['GR_roll_mean_10'] = df['GR'].rolling(window=10, min_periods=1).mean()
    df['GR_roll_std_10'] = df['GR'].rolling(window=10, min_periods=1).std().fillna(0)
    
    # 3. 直近50ステップの移動平均（大きな地層の変化）
    df['GR_roll_mean_50'] = df['GR'].rolling(window=50, min_periods=1).mean()
    
    return df
# -------------------------------------------------------------

print("学習データを読み込み、特徴量を作成しています...")
train_files = glob.glob(f'{base_path}/train/*__horizontal_well.csv')
train_list = []
for f in train_files:
    well_id = os.path.basename(f).split('__')[0]
    df = pd.read_csv(f)
    df['well_id'] = well_id
    df = create_features(df) # ★ここで波のトレンドを追加！
    train_list.append(df)
train_df = pd.concat(train_list, ignore_index=True)

print("テストデータを読み込み、特徴量を作成しています...")
test_files = glob.glob(f'{base_path}/test/*__horizontal_well.csv')
test_list = []
for f in test_files:
    well_id = os.path.basename(f).split('__')[0]
    df = pd.read_csv(f)
    df['well_id'] = well_id
    df['id'] = well_id + '_' + df.index.astype(str)
    df = create_features(df) # ★ここでも波のトレンドを追加！
    test_list.append(df)
test_df = pd.concat(test_list, ignore_index=True)

print("モデルの学習を開始します...")
# ★ 特徴量に新しく作った列を追加！
features = [
    'MD', 'X', 'Y', 'Z', 'GR', 
    'GR_diff', 'Z_diff', 'GR_roll_mean_10', 'GR_roll_std_10', 'GR_roll_mean_50'
]
target = 'TVT'

train_df[features] = train_df[features].fillna(0)
test_df[features] = test_df[features].fillna(0)
train_df = train_df.dropna(subset=[target])

# GroupKFoldによるモデル学習
gkf = GroupKFold(n_splits=5)
models = []
for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df[target], train_df['well_id'])):
    X_tr, y_tr = train_df.iloc[train_idx][features], train_df.iloc[train_idx][target]
    X_va, y_va = train_df.iloc[val_idx][features], train_df.iloc[val_idx][target]
    
    model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)])
    models.append(model)

print("予測と提出ファイルの作成を行っています...")
preds = np.zeros(len(test_df))
for model in models:
    preds += model.predict(test_df[features]) / len(models)

test_df['predicted_tvt'] = preds

sub = pd.read_csv(f'{base_path}/sample_submission.csv')
sub = sub.drop(columns=['tvt']).merge(test_df[['id', 'predicted_tvt']], on='id', how='left')
sub = sub.rename(columns={'predicted_tvt': 'tvt'})
sub['tvt'] = sub['tvt'].fillna(0.0)

sub[['id', 'tvt']].to_csv('submission.csv', index=False)
print("完了しました！submission.csv が作成されました。")